In [1]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
os.chdir("/content/drive/MyDrive/Colab Notebooks")

In [4]:
train_data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/samsum-train.csv")
val_data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/samsum-validation.csv")
test_data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/samsum-test.csv")

In [5]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [6]:
train_data.shape

(14732, 3)

In [7]:
val_data.shape

(818, 3)

## Data Pre-processing

In [8]:
import re
import pandas as pd

def clean_data(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"\r\n", " ", text)  # Replace newlines with space
    text = re.sub(r"\s+", " ", text)  # Remove extra whitespace
    text = re.sub(r"<.*?>", " ", text)  # Remove HTML tags
    return text.strip().lower()

In [9]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [10]:
# Tokenization
tokenizer = T5Tokenizer.from_pretrained("t5-small")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
def tokenize_data(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", truncation=True, max_length=512)
    targets = tokenizer(data["summary"], padding="max_length", truncation=True, max_length=150)

    inputs["labels"] = targets["input_ids"]
    return inputs

In [12]:
train_dataset = train_data.apply(tokenize_data, axis=1).tolist()
val_dataset = val_data.apply(tokenize_data, axis=1).tolist()

In [13]:
train_dataset[0]

{'input_ids': [183, 232, 9, 10, 3, 23, 13635, 5081, 5, 103, 25, 241, 128, 58, 3, 12488, 651, 10, 417, 55, 183, 232, 9, 10, 3, 23, 31, 195, 830, 25, 5721, 3, 10, 18, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [14]:
# inputs ids - dialogue => token ids
# attention mask - dialogue => 1 for actual tokens, 0 for padding
# labels - target => summary token

In [15]:
len(train_dataset[0]["input_ids"])

512

In [16]:
type(train_dataset)
type(val_dataset)

list

## Working with our model

In [17]:
# NLP => Generation tasks - T5, BART, Pegasus
model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [18]:
#fine-tuning
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Device : ", device)
model.to(device)

Device :  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [22]:
# Training arguments
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Colab Notebooks/results",
    num_train_epochs=10,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    warmup_steps=500
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [23]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [24]:
# Training the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.392938,0.341025
2,0.358447,0.329718
3,0.353384,0.323703
4,0.339680,0.321844


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=18420, training_loss=0.41467623218780747, metrics={'train_runtime': 7572.234, 'train_samples_per_second': 19.455, 'train_steps_per_second': 2.433, 'total_flos': 1.993855419285504e+16, 'train_loss': 0.41467623218780747, 'epoch': 10.0})

In [25]:
# model load => fine tune => save the model
model.save_pretrained("/content/drive/MyDrive/Colab Notebooks/models/t5-summary-model")
tokenizer.save_pretrained("/content/drive/MyDrive/Colab Notebooks/models/t5-summary-tokenizer")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Colab Notebooks/models/t5-summary-tokenizer/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/models/t5-summary-tokenizer/tokenizer.json')

In [26]:
model = T5ForConditionalGeneration.from_pretrained("/content/drive/MyDrive/Colab Notebooks/models/t5-summary-model")
tokenizer = T5Tokenizer.from_pretrained("/content/drive/MyDrive/Colab Notebooks/models/t5-summary-tokenizer")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

### Test the core logic for summarization

In [27]:
def summarize_dialogue(dialogue, model, tokenizer):
  # tokenize
  inputs = tokenizer(
      dialogue,
      max_length=512,
      padding="max_length",
      truncation=True,
      return_tensors="pt"
  ).to(device)

  # generate the summary
  model.to(device)
  targets = model.generate(
      input_ids=inputs["input_ids"],
      attention_mask=inputs["attention_mask"],
      max_length=150,
      num_beams=4,
      early_stopping=True
  )

  # tokens ids convert to summary => decoding
  summary = tokenizer.decode(
      targets[0],
      skip_special_tokens=True,
      clean_up_tokenization_spaces=False
  )

  return summary

In [31]:
test_dialogue = """
FM: We need to finalize today’s agenda. The public has been pressing hard for relief on essential goods.

PA: Yes, Minister. The proposal is to reduce GST on items like food staples, medicines, and hygiene products.

FM: What’s the current rate on these essentials?

PA: Most are taxed at 5%. The recommendation is to bring them down to 3%, or even exempt a few categories entirely.

FM: Exemption sounds appealing politically, but it cuts into revenue. How much would we lose if we go to 3%?

PA: Preliminary estimates suggest a shortfall of ₹12,000 crore annually. However, this could be offset by increased consumption and compliance.

FM: [thoughtful] So the argument is that lower taxes encourage broader participation and reduce evasion.

PA: Exactly. Plus, it directly benefits lower-income households. Cheaper essentials mean more disposable income for other needs.

FM: What about industry feedback? Have manufacturers weighed in?

PA: They’re supportive. Lower GST reduces their pricing pressure and could boost demand. But they want clarity — no frequent changes.

FM: Stability is key. We can’t keep tinkering with rates every quarter.

PA: Agreed. Another point — healthcare groups are lobbying for zero GST on life-saving drugs. That could be a strong social signal.

FM: [nodding] True, but we must balance fiscal responsibility. Perhaps a phased reduction: start with 3%, review impact after six months, then consider exemptions.

PA: That would show responsiveness without risking a sudden revenue shock.

FM: Good. Let’s draft the agenda accordingly:

Proposal to reduce GST on food staples and hygiene products to 3%.

Review mechanism after six months.

Special consideration for life-saving drugs — possible exemption in phase two.

PA: I’ll prepare the briefing note. Shall we also include a communication strategy?

FM: Absolutely. Public perception matters as much as policy. Frame this as relief for households, not just a tax adjustment.

PA: Understood, Minister. I’ll ensure the messaging highlights affordability and fairness.

FM: Excellent. Let’s move forward. This could be one of the most impactful reforms of the year.
"""

summary = summarize_dialogue(test_dialogue, model, tokenizer)
print(summary)


the public has been pressing for relief on essential goods. the current rate on food staples, medicines, and hygiene products is 5%. the recommendation is to bring them down to 3%, or even exempt a few categories entirely.
